# max_ops

> MAX Engine integration for CPU and GPU compute.
> Unified Mojo kernels compile to both targets - write once, run anywhere.

In [1]:
#| default_exp max_ops

In [2]:
#| export
from pathlib import Path
import numpy as np
from typing import Optional, Tuple, Dict, List, Any, Literal, Union

from max import engine, driver
from max.graph import Graph, TensorType, ops, DeviceRef
from max.dtype import DType
from max.experimental import tensor as mx_tensor  # Experimental eager tensor API

# Type alias for device selection
DeviceType = Literal["cpu", "gpu", "auto"]
Scalar = Union[int, float, bool]


In [3]:
#| export
# Kernel path - hardcoded relative to package
KERNELS_PATH = Path(__file__).parent.parent / "kernels" if '__file__' in dir() else Path.cwd().parent / "kernels"

In [4]:
#| export
class MAXSession:
    """
    Session manager for MAX Engine compute on CPU or GPU.
    
    Manages device selection, session creation, and graph caching.
    Same Mojo kernel code compiles to both CPU and GPU targets.
    
    Example:
        >>> cpu_sess = MAXSession.get("cpu")
        >>> gpu_sess = MAXSession.get("gpu")
        >>> result = max_sum(arr, device="cpu")  # Uses CPU session
        >>> result = max_sum(arr, device="gpu")  # Uses GPU session
    """
    _instances: Dict[str, 'MAXSession'] = {}
    
    def __init__(self, device_type: DeviceType = "auto"):
        """
        Create a session for the specified device.
        
        Args:
            device_type: "cpu", "gpu", or "auto" (GPU if available, else CPU)
        """
        # Resolve device
        if device_type == "auto":
            device_type = "gpu" if driver.accelerator_count() > 0 else "cpu"
        
        self.device_type = device_type
        
        if device_type == "gpu":
            if driver.accelerator_count() == 0:
                raise RuntimeError("No GPU available, use device='cpu'")
            self.device = driver.Accelerator()
            self.device_ref = DeviceRef.GPU()
        else:
            self.device = driver.CPU()
            self.device_ref = DeviceRef.CPU()
        
        # Create session for this device
        self.session = engine.InferenceSession(devices=[self.device])
        
        # Graph cache: (op_name, shape, dtype) -> compiled model
        self._graph_cache: Dict[Tuple[str, tuple, str], Any] = {}
    
    @classmethod
    def get(cls, device: DeviceType = "auto") -> 'MAXSession':
        """Get or create a session for the specified device."""
        # Resolve auto
        if device == "auto":
            device = "gpu" if driver.accelerator_count() > 0 else "cpu"
        
        if device not in cls._instances:
            cls._instances[device] = cls(device)
        return cls._instances[device]
    
    @classmethod
    def reset(cls):
        """Reset all sessions (for testing)."""
        cls._instances = {}
    
    def _cache_key(self, op_name: str, shape: tuple, dtype: str) -> Tuple[str, tuple, str]:
        return (op_name, shape, dtype)
    
    def _get_or_compile(self, op_name: str, shape: tuple, dtype: str, 
                        build_graph_fn) -> Any:
        """Get cached model or compile new one."""
        key = self._cache_key(op_name, shape, dtype)
        if key not in self._graph_cache:
            graph = build_graph_fn()
            self._graph_cache[key] = self.session.load(graph)
        return self._graph_cache[key]
    
    def to_tensor(self, arr: np.ndarray) -> driver.Tensor:
        """Convert numpy array to MAX tensor on this session's device."""
        arr = np.ascontiguousarray(arr, dtype=np.float32)
        tensor = driver.Tensor.from_numpy(arr)
        if self.device_type == "gpu":
            tensor = tensor.to(self.device)
        return tensor
    
    def from_tensor(self, tensor: driver.Tensor) -> np.ndarray:
        """Convert MAX tensor back to numpy."""
        if self.device_type == "gpu":
            tensor = tensor.to(driver.CPU())
        return tensor.to_numpy()

In [5]:
#| export
def _get_device(device: DeviceType = "auto") -> driver.Device:
    """Get the appropriate device based on selection."""
    if device == "auto":
        device = "gpu" if driver.accelerator_count() > 0 else "cpu"
    if device == "gpu":
        if driver.accelerator_count() == 0:
            raise RuntimeError("No GPU available")
        return driver.Accelerator()
    return driver.CPU()


def _to_mx_tensor(arr: np.ndarray, device: DeviceType = "auto") -> mx_tensor.Tensor:
    """Convert numpy array to MAX experimental tensor on specified device."""
    arr = np.ascontiguousarray(arr, dtype=np.float32)
    t = mx_tensor.Tensor.from_dlpack(arr)
    dev = _get_device(device)
    if isinstance(dev, driver.Accelerator):
        t = t.to(dev)
    return t


def _from_mx_tensor(t: mx_tensor.Tensor) -> np.ndarray:
    """Convert MAX tensor back to numpy (transfers to CPU if needed)."""
    if hasattr(t.device, '__class__') and 'Accelerator' in t.device.__class__.__name__:
        t = t.to(driver.CPU())
    # Use DLPack protocol to get numpy
    return np.from_dlpack(t)


## Eager Tensor Operations

Using `max.experimental.tensor` for direct operations without Graph compilation.
These work on both CPU and GPU with automatic optimization.

In [6]:
#| export
def max_mean(arr: np.ndarray, device: DeviceType = "auto") -> float:
    """Compute mean using MAX Engine experimental tensor API."""
    t = _to_mx_tensor(arr, device)
    result = t.mean(axis=None)
    return float(result.item())


def max_min(arr: np.ndarray, device: DeviceType = "auto") -> float:
    """Compute min using MAX Engine experimental tensor API."""
    t = _to_mx_tensor(arr, device)
    # Use reduction - min along all axes
    # Note: If min() not available, we'll fall back
    try:
        result = t.min(axis=None) if hasattr(t, 'min') else -t.max(axis=None)  # Fallback
    except AttributeError:
        # Fallback to numpy if min not in experimental API
        return float(np.min(arr))
    return float(result.item())


def max_max(arr: np.ndarray, device: DeviceType = "auto") -> float:
    """Compute max using MAX Engine experimental tensor API."""
    t = _to_mx_tensor(arr, device)
    result = t.max(axis=None)
    return float(result.item())


def max_count(arr: np.ndarray, device: DeviceType = "auto") -> int:
    """Count non-null elements."""
    # For now, count all elements (null handling would need mask)
    return len(arr)


def max_std(arr: np.ndarray, device: DeviceType = "auto") -> float:
    """Compute standard deviation using MAX Engine."""
    t = _to_mx_tensor(arr, device)
    try:
        # Experimental API may have stdev
        result = t.stdev(axis=None) if hasattr(t, 'stdev') else t.std(axis=None)
        return float(result.item())
    except AttributeError:
        # Fallback to computing from mean
        mean_val = max_mean(arr, device)
        variance = max_mean((arr - mean_val) ** 2, device)
        return float(np.sqrt(variance))


In [7]:
# Test aggregations
arr = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32)
print(f"Array: {arr}")
print(f"max_mean (CPU): {max_mean(arr, 'cpu')}, expected: {arr.mean()}")
print(f"max_max (CPU): {max_max(arr, 'cpu')}, expected: {arr.max()}")
print(f"max_min (CPU): {max_min(arr, 'cpu')}, expected: {arr.min()}")
print(f"max_std (CPU): {max_std(arr, 'cpu')}, expected: {arr.std()}")

# GPU test
if driver.accelerator_count() > 0:
    print(f"\nmax_mean (GPU): {max_mean(arr, 'gpu')}")
    print(f"max_max (GPU): {max_max(arr, 'gpu')}")


Array: [1. 2. 3. 4. 5.]


TypeError: '<' not supported between instances of 'NoneType' and 'int'

In [ ]:
#| export
def max_sum(arr: np.ndarray, device: DeviceType = "auto") -> float:
    """
    Compute sum using MAX Engine on CPU or GPU.
    
    Same compiled kernel runs on either target.
    
    Args:
        arr: NumPy array (will be converted to float32)
        device: "cpu", "gpu", or "auto"
        
    Returns:
        Sum as float
    """
    sess = MAXSession.get(device)
    tensor = sess.to_tensor(arr)
    
    def build_graph():
        dtype_max = DType.float32
        with Graph("sum", input_types=[TensorType(dtype_max, tensor.shape, sess.device_ref)]) as g:
            x = g.inputs[0]
            result = ops.sum(x)
            g.output(result)
        return g
    
    model = sess._get_or_compile("sum", tensor.shape, "float32", build_graph)
    result = model.execute(tensor)[0]
    return float(sess.from_tensor(result).item())


# Backward compatibility alias
def gpu_sum(arr: np.ndarray) -> float:
    """Deprecated: Use max_sum(arr, device='gpu') instead."""
    return max_sum(arr, device="gpu")

In [ ]:
# Test max_sum on both CPU and GPU
import numpy as np

arr = np.array([1, 2, 3, 4, 5], dtype=np.float32)

# CPU
cpu_result = max_sum(arr, device="cpu")
print(f"max_sum (CPU): {cpu_result}")

# GPU
gpu_result = max_sum(arr, device="gpu")
print(f"max_sum (GPU): {gpu_result}")

print(f"Expected: {arr.sum()}")
print(f"Match: {cpu_result == gpu_result == arr.sum()}")

In [ ]:
#| export
def max_masked_sum(mask: np.ndarray, values: np.ndarray, 
                   device: DeviceType = "auto") -> float:
    """
    Compute masked sum: sum(mask * values) using MAX Engine.
    
    Args:
        mask: Boolean or float mask array
        values: Values to sum where mask is true/nonzero
        device: "cpu", "gpu", or "auto"
        
    Returns:
        Masked sum as float
    """
    sess = MAXSession.get(device)
    
    # Convert to float32
    mask_f = mask.astype(np.float32) if mask.dtype != np.float32 else mask
    vals_f = values.astype(np.float32) if values.dtype != np.float32 else values
    
    mask_tensor = sess.to_tensor(mask_f)
    vals_tensor = sess.to_tensor(vals_f)
    
    def build_graph():
        dtype_max = DType.float32
        shape = mask_tensor.shape
        with Graph("masked_sum", input_types=[
            TensorType(dtype_max, shape, sess.device_ref),
            TensorType(dtype_max, shape, sess.device_ref),
        ]) as g:
            m, v = g.inputs
            product = ops.mul(m, v)
            result = ops.sum(product)
            g.output(result)
        return g
    
    model = sess._get_or_compile("masked_sum", mask_tensor.shape, "float32", build_graph)
    result = model.execute(mask_tensor, vals_tensor)[0]
    return float(sess.from_tensor(result).item())


# Backward compatibility alias
def gpu_masked_sum(mask: np.ndarray, values: np.ndarray) -> float:
    """Deprecated: Use max_masked_sum(mask, values, device='gpu') instead."""
    return max_masked_sum(mask, values, device="gpu")

In [ ]:
# Test max_masked_sum on both CPU and GPU
mask = np.array([1, 0, 1, 0, 1], dtype=np.float32)
values = np.array([10, 20, 30, 40, 50], dtype=np.float32)

cpu_result = max_masked_sum(mask, values, device="cpu")
gpu_result = max_masked_sum(mask, values, device="gpu")

print(f"max_masked_sum (CPU): {cpu_result}")
print(f"max_masked_sum (GPU): {gpu_result}")
print(f"Expected: {(mask * values).sum()}")

In [ ]:
#| export  
def gpu_masked_sum_kernel(mask: np.ndarray, values: np.ndarray, 
                          kernels_path: Optional[Path] = None) -> float:
    """
    GPU-accelerated masked sum using custom Mojo kernel.
    
    Uses the fused masked_sum_simple kernel with warp reduction.
    Falls back to built-in ops if kernel not found.
    
    Args:
        mask: Boolean or float mask array
        values: Values to sum where mask is true/nonzero
        kernels_path: Path to kernels directory (default: ../kernels)
        
    Returns:
        Masked sum as float
    """
    sess = MAXSession.get()
    
    if kernels_path is None:
        kernels_path = Path.cwd().parent / "kernels"
    
    if not kernels_path.exists():
        # Fallback to built-in ops
        return gpu_masked_sum(mask, values)
    
    # Convert to float32
    mask_f = np.ascontiguousarray(mask, dtype=np.float32)
    vals_f = np.ascontiguousarray(values, dtype=np.float32)
    
    mask_tensor = sess.to_tensor(mask_f)
    vals_tensor = sess.to_tensor(vals_f)
    
    # Output size = ceil(n / 32) for warp partial sums
    n = len(mask_f)
    output_size = (n + 31) // 32
    
    def build_graph():
        dtype_max = DType.float32
        with Graph("masked_sum_kernel", 
                   input_types=[
                       TensorType(dtype_max, mask_tensor.shape, sess.device_ref),
                       TensorType(dtype_max, vals_tensor.shape, sess.device_ref),
                   ],
                   custom_extensions=[kernels_path]) as g:
            m, v = g.inputs
            # Call custom kernel
            partial_sums = ops.custom(
                name="masked_sum_simple",
                device=sess.device_ref,
                values=[m, v],
                out_types=[TensorType(dtype_max, (output_size,), sess.device_ref)],
            )[0]
            # Sum the warp partial sums
            result = ops.sum(partial_sums)
            g.output(result)
        return g
    
    cache_key = f"masked_sum_kernel_{kernels_path}"
    model = sess._get_or_compile(cache_key, mask_tensor.shape, "float32", build_graph)
    result = model.execute(mask_tensor, vals_tensor)[0]
    return float(sess.from_tensor(result).item())

In [ ]:
# Test custom kernel masked sum
kernels_path = Path.cwd().parent / "kernels"
print(f"Kernels path: {kernels_path}")
print(f"Exists: {kernels_path.exists()}")

if kernels_path.exists():
    mask = np.array([1, 0, 1, 0, 1], dtype=np.float32)
    values = np.array([10, 20, 30, 40, 50], dtype=np.float32)
    
    result = gpu_masked_sum_kernel(mask, values, kernels_path)
    print(f"gpu_masked_sum_kernel = {result}")
    print(f"Expected: {(mask * values).sum()}")

In [ ]:
# Benchmark: MAX CPU vs MAX GPU vs NumPy
import time

sizes = [100_000, 1_000_000, 10_000_000]

print("Benchmark: numpy vs MAX CPU vs MAX GPU")
print("=" * 65)

for n in sizes:
    arr = np.random.rand(n).astype(np.float32)
    
    # Warm up (includes compilation)
    _ = max_sum(arr, device="cpu")
    _ = max_sum(arr, device="gpu")
    _ = arr.sum()
    
    # Time numpy
    t0 = time.perf_counter()
    for _ in range(10):
        np_result = arr.sum()
    np_time = (time.perf_counter() - t0) / 10 * 1000
    
    # Time MAX CPU
    t0 = time.perf_counter()
    for _ in range(10):
        cpu_result = max_sum(arr, device="cpu")
    cpu_time = (time.perf_counter() - t0) / 10 * 1000
    
    # Time MAX GPU
    t0 = time.perf_counter()
    for _ in range(10):
        gpu_result = max_sum(arr, device="gpu")
    gpu_time = (time.perf_counter() - t0) / 10 * 1000
    
    print(f"n={n:>10,}: numpy={np_time:>7.2f}ms | MAX CPU={cpu_time:>7.2f}ms | MAX GPU={gpu_time:>7.2f}ms")

In [ ]:
#| export
def get_device_info() -> dict:
    """Get information about available MAX devices and sessions."""
    has_gpu = driver.accelerator_count() > 0
    
    info = {
        "accelerator_count": driver.accelerator_count(),
        "sessions": list(MAXSession._instances.keys()),
    }
    
    for device, sess in MAXSession._instances.items():
        info[f"{device}_cache_size"] = len(sess._graph_cache)
    
    return info

## Fused Group Aggregations

Single-pass aggregations per group using MAX Engine.

In [ ]:
#| export
def max_fused_group_sum(
    group_ids: np.ndarray, 
    values: np.ndarray, 
    n_groups: int,
    device: DeviceType = "auto",
    kernels_path: Optional[Path] = None,
) -> np.ndarray:
    """
    Compute sum per group in a single pass using MAX Engine.
    
    Uses a fused kernel that processes all groups simultaneously,
    avoiding multiple passes over the data.
    
    Args:
        group_ids: Integer array of group IDs (0 to n_groups-1)
        values: Values to sum per group
        n_groups: Number of unique groups
        device: "cpu", "gpu", or "auto"
        kernels_path: Path to kernels directory
        
    Returns:
        Array of sums, one per group [n_groups]
    """
    sess = MAXSession.get(device)
    
    if kernels_path is None:
        kernels_path = KERNELS_PATH
    
    # Convert inputs
    gids = np.ascontiguousarray(group_ids, dtype=np.int32)
    vals = np.ascontiguousarray(values, dtype=np.float32)
    n = len(vals)
    
    # For now, use simple mask-per-group approach with MAX Engine
    # (Custom fused kernel would be faster but requires more setup)
    results = np.zeros(n_groups, dtype=np.float32)
    
    for g in range(n_groups):
        mask = (gids == g).astype(np.float32)
        results[g] = max_masked_sum(mask, vals, device=device)
    
    return results

In [ ]:
#| export
def max_fused_multi_agg(
    group_ids: np.ndarray,
    columns: Dict[str, np.ndarray],
    agg_specs: List[Tuple[str, str, str]],  # (output_name, column, op)
    n_groups: int,
    device: DeviceType = "auto",
) -> Dict[str, np.ndarray]:
    """
    Compute multiple aggregations per group.
    
    This is the building block for MXGroupBy.agg() with fused execution.
    
    Args:
        group_ids: Integer array of group IDs
        columns: Dict of column_name -> values array
        agg_specs: List of (output_name, column, op) tuples
        n_groups: Number of unique groups
        device: Device selection
        
    Returns:
        Dict of output_name -> result array (length n_groups)
    """
    sess = MAXSession.get(device)
    gids = np.ascontiguousarray(group_ids, dtype=np.int32)
    
    results = {}
    
    # Group by aggregation type for potential fusion
    for output_name, col_name, op in agg_specs:
        vals = np.ascontiguousarray(columns[col_name], dtype=np.float32)
        
        if op == 'sum':
            results[output_name] = max_fused_group_sum(gids, vals, n_groups, device)
        elif op == 'count':
            # Count is sum of ones
            ones = np.ones_like(vals)
            results[output_name] = max_fused_group_sum(gids, ones, n_groups, device)
        elif op == 'mean':
            sums = max_fused_group_sum(gids, vals, n_groups, device)
            counts = max_fused_group_sum(gids, np.ones_like(vals), n_groups, device)
            results[output_name] = sums / np.maximum(counts, 1)
        elif op in ('min', 'max'):
            # Fallback to numpy for min/max
            result = np.zeros(n_groups, dtype=np.float32)
            for g in range(n_groups):
                masked = vals[gids == g]
                if len(masked) > 0:
                    result[g] = np.min(masked) if op == 'min' else np.max(masked)
                else:
                    result[g] = np.nan
            results[output_name] = result
        else:
            raise ValueError(f"Unknown aggregation: {op}")
    
    return results

In [ ]:
# Check sessions and cache
print(get_device_info())

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()